# Module 5: Datasets and Experiments

> Self-directed module, ~30 min.

Offline evaluations score the agent on demand against a fixed dataset. Three patterns, ordered broadest to most granular:

1. **Final-response** — does the end-to-end answer satisfy the rubric? (LLM-as-judge)
2. **Single-step** — did the agent make the correct first decision? (does the first tool call match the expected one)
3. **Trajectory** — does the full sequence of tool calls match the expected trajectory? (exact match plus extra/missing step counts)

All three run against the same dataset, just with different run functions and evaluators.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import time
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")

from utils.models import model
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import SystemMessage, HumanMessage
from collections import Counter
from typing import Any


## Part 1. Dataset

Five examples covering the agent's main paths: profile-only, web-research-only, profile + research, and the unknown-client fall-through. Each example carries three reference fields so the same dataset supports all three eval types:

- `reference_answer` — a success rubric (not an expected response verbatim) for the LLM judge
- `expected_first_tool` — the first tool call we expect the agent to make
- `trajectory` — the full sequence of tool calls expected


In [ ]:
examples = [
    {
        "inputs": {"query": "What does Smith Family Office hold?"},
        "outputs": {
            "reference_answer": "Lists Smith Family Office's top holdings (NVDA, GOOGL, BRK.B) with weights and the aggressive-growth risk profile.",
            "expected_first_tool": "get_client_profile",
            "trajectory": ["get_client_profile"],
        },
    },
    {
        "inputs": {"query": "Summarize recent news on Apple. Search at most once."},
        "outputs": {
            "reference_answer": "A brief, cited summary of recent Apple news from a web search.",
            "expected_first_tool": "task",
            "trajectory": ["task"],
        },
    },
    {
        "inputs": {"query": "Prep me for my meeting with Acme Pension Fund. Cover one major holding. Search at most once."},
        "outputs": {
            "reference_answer": "Acme Pension Fund's holdings and risk profile, plus recent news on one major holding (e.g., AAPL, MSFT, JPM, or XOM), with at least one cited source.",
            "expected_first_tool": "get_client_profile",
            "trajectory": ["get_client_profile", "task"],
        },
    },
    {
        "inputs": {"query": "Look up TechCorp Treasury, then research one of their holdings briefly. Search at most once."},
        "outputs": {
            "reference_answer": "TechCorp Treasury's conservative, fixed-income-heavy profile, plus recent context on one holding.",
            "expected_first_tool": "get_client_profile",
            "trajectory": ["get_client_profile", "task"],
        },
    },
    {
        "inputs": {"query": "Who is the client 'unknown-fund'?"},
        "outputs": {
            "reference_answer": "Tells the user the client is not in the system and lists the known client IDs.",
            "expected_first_tool": "get_client_profile",
            "trajectory": ["get_client_profile"],
        },
    },
]

dataset_name = "client-research-evals"

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")


## Part 2. Final-Response Evaluation

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Run the agent end-to-end, capture its final response, score it against the rubric with an LLM judge. The rubric is success criteria, not verbatim expected text — the judge prompt tells the model to grade against criteria, not literal match.


In [ ]:
def run_agent_final(inputs: dict) -> dict:
    """Target function: run the agent and return its final response.

    Also concatenates any files the agent wrote so the judge sees actual content,
    not just the agent's 'saved!' confirmation message.
    """
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    response = result["messages"][-1].content

    file_dump = []
    for path, file_data in (result.get("files") or {}).items():
        content = file_data
        if isinstance(file_data, dict) and "content" in file_data:
            content = file_data["content"]
        if isinstance(content, list):
            content = "\n".join(content)
        file_dump.append(f"--- {path} ---\n{content}")
    if file_dump:
        response += "\n\nFiles written:\n" + "\n\n".join(file_dump)

    return {"response": response}


In [ ]:
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the rubric."""
    score: Annotated[bool, ..., "True if the response meets the success rubric"]
    reasoning: Annotated[str, ..., "One short sentence of reasoning"]

correctness_judge_prompt = """You are grading a client research assistant.

You will see the user's request, the assistant's final response, and a RUBRIC describing success.

The rubric is success criteria, not the expected response text. Mark score=True if the response
demonstrates the criteria were met — even if worded differently from the rubric. Mark score=False
only if the response clearly missed the task, hallucinated facts (especially client identities),
or refused without good reason.

Give one short sentence of reasoning either way.
"""

judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "correctness", "score": int(grade["score"]), "comment": grade["reasoning"]}


In [ ]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


## Part 3. Single-Step Evaluation

<img src="../images/single-step.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Final-response evals are end-to-end; single-step evals zoom in on one decision. For the client research agent, the most informative single decision is the **first tool call**: when a named client is in the query, did the agent call `get_client_profile` first? When the query is open-ended research, did it delegate to the `research-agent` subagent via `task`?

The run function captures only the first tool call; the evaluator checks it against `expected_first_tool`.


In [ ]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_first_tool(inputs: dict) -> dict:
    """Single-step target: capture only the first tool the agent called."""
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    tool_calls = extract_tool_calls(result["messages"])
    return {"first_tool": tool_calls[0] if tool_calls else None}

def first_tool_match(outputs, reference_outputs):
    """Exact match between the agent's first tool call and the expected one."""
    return {
        "key": "first_tool_match",
        "score": int(outputs["first_tool"] == reference_outputs["expected_first_tool"]),
    }


In [ ]:
results = client.evaluate(
    run_agent_first_tool,
    data=dataset_name,
    evaluators=[first_tool_match],
    experiment_prefix="single-step",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


## Part 4. Trajectory Evaluation

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Trajectory evaluations score the full sequence of tool calls, not just the first or the final output. Three metrics:

- **Exact match** — does the trajectory equal the reference, in order?
- **Extra steps** — how many tool calls did the agent make that weren't in the reference?
- **Missing steps** — how many expected tool calls did the agent skip?

Extra and missing surface drift even when exact match is too strict — useful when the agent has a few valid paths.


In [ ]:
def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


In [ ]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


## Recap

| Pattern | What it scores | Reference field used |
|---|---|---|
| **Final-response** | End-to-end response against a success rubric (LLM-as-judge) | `reference_answer` |
| **Single-step** | First tool call decision | `expected_first_tool` |
| **Trajectory** | Full tool-call sequence (exact + extra + missing) | `trajectory` |

All three reuse the same dataset; only the run function and evaluator change. The LangSmith UI groups experiments by `experiment_prefix` so iterations are easy to compare side-by-side.
